In [7]:
import numpy as np
from scipy.integrate import quad, dblquad
from scipy.integrate import fixed_quad


In [8]:

# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

ensemble_parameters = {
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        }
    }
}

ensemble_totem = 'totem'
log_model_type = 'log'


In [9]:

# define model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, a2, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


In [10]:
def full_int(mg, a1, a2, m2_func, q2_val, k_max=10):
    """
    Calcula a integral sobre k e phi sem dependência de sqrt_s
    
    Parâmetros:
    -----------
    k_max : float, opcional
        Limite superior para integração em k (default = 10 GeV)
    """
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []
    
    for q2 in q2_val:
        def integrand(k, phi):
            """Integrando em coordenadas polares (k, phi)"""
            # O fator k vem do Jacobiano em coordenadas polares
            return k * (T_1(k, q2, phi, mg, a1, a2, m2_func) - T_2(k, q2, phi, mg, a1, a2, m2_func))
        
        # Integração dupla: k de 0 a k_max, phi de 0 a 2π
        result, _ = dblquad(
            integrand,
            0, 2 * np.pi,  # limites para phi
            lambda phi: 0,  # limite inferior para k
            lambda phi: k_max,  # limite superior para k
            epsabs=1e-16,
            epsrel=1e-8
        )
        
        results.append(result)
    
    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [14]:
print(full_int(0.380, 1.491, 2.77, m2_log, 1.0, k_max=1000))


0.0006537987656793318


In [17]:
def full_int(mg, a1, a2, m2_func, k_max=10):
    """
    Calcula a integral tripla em q, phi e k
    
    Parâmetros:
    -----------
    k_max : float, opcional
        Limite superior para integração em k (default = 10 GeV)
    """
    
    def integrand_q(q):
        """Integrando externo em q"""
        
        def integrand(k, phi):
            """Integrando em coordenadas polares (k, phi)"""
            # Agora usamos q^2 internamente
            q2 = q**2
            return k * (T_1(k, q2, phi, mg, a1, a2, m2_func) -
                        T_2(k, q2, phi, mg, a1, a2, m2_func))
        
        # Integração dupla: k de 0 a k_max, phi de 0 a 2π
        result, _ = dblquad(
            integrand,
            0, 2 * np.pi,      # limites para phi
            lambda phi: 0,     # limite inferior para k
            lambda phi: k_max, # limite superior para k
            epsabs=1e-16,
            epsrel=1e-8
        )
        
        return result

    # Integração externa em q de 0 a 5
    final_result, _ = quad(
        integrand_q,
        0, 5,
        epsabs=1e-16,
        epsrel=1e-8
    )
    
    return final_result

print(full_int(0.380, 1.491, 2.77, m2_log, k_max=10))


2.4253833573777275
